# Plan with CLIP-ViT Planner on MiniGrid

Load a trained RSSM world model and use a CLIP-ViT-based planner to select actions via heuristic candidate sampling and discounted cosine-similarity scoring.

## 1. Setup

In [ ]:
!pip install gymnasium minigrid matplotlib pyyaml Pillow tqdm open-clip-torch -q

import sys
!git clone https://github.com/mihalko711/tbank_intro_problem.git
sys.path.insert(0, 'tbank_intro_problem')
%cd tbank_intro_problem

## 2. Imports, config, env, checkpoint

In [ ]:
import os
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm.notebook import tqdm

from src import (
    RSSMWorldModel, collect_episode, evaluate,
    get_env_properties, make_minigrid_env, seed_everything,
)
from src.planner import CLIPScorer, Planner, HeuristicCandidates

CHECKPOINT_PATH = "/kaggle/input/tbank-wm-checkpoint/MiniGrid-Empty-6x6-v0_default_final.pth"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

with open('configs/minigrid_default.yml') as f:
    config = yaml.safe_load(f)
rssm_cfg = config['rssm']
seed_everything(config['seed'])

env = make_minigrid_env(config['environment_name'], seed=config['seed'])
env_eval = make_minigrid_env(config['environment_name'], seed=config['seed'] + 999)
obs_shape, action_size = get_env_properties(env)
print(f'Obs shape: {obs_shape}, action size: {action_size}')

rssm = RSSMWorldModel(obs_shape, action_size, rssm_cfg, device)
rssm.load_checkpoint(CHECKPOINT_PATH)
print(f'Checkpoint loaded: {CHECKPOINT_PATH}')

## 3. Create CLIP-ViT Planner

In [ ]:
clip_scorer = CLIPScorer(device, rssm)
planner = Planner(
    rssm, clip_scorer,
    num_candidates=64,
    horizon=rssm_cfg['imagination_horizon'],
    gamma=0.99,
)
print('Planner (CLIP-ViT) ready')
print(f'Heuristic candidates: {planner.num_candidates} x {planner.horizon} steps')

## 4. Visualise heuristic candidates

Take a starting observation from a random episode, decode the imagined trajectories for the best, median, and worst candidate.

In [ ]:
from src.utils import symexp

@torch.no_grad()
def visualize_candidates(rssm, planner, env, action_size, n_cols=15, save_path=None):
    rec, lat = rssm.reset_state()
    action = torch.zeros(1, action_size, device=rssm.device)
    obs = env.reset()
    rec = rssm.recurrent_model(rec, lat, action)
    encoded = rssm.encoder(torch.from_numpy(obs).float().unsqueeze(0).to(rssm.device))
    lat, _ = rssm.posterior_net(torch.cat((rec, encoded.view(1, -1)), -1))

    candidates = planner._candidate_sampler.sample()
    trajectories = rssm.imagine_rollouts(rec, lat, candidates)
    scores = clip_scorer.score_rollouts(trajectories, planner.gamma).reshape(-1)

    order = scores.argsort(descending=True)
    idxs = [order[0].item(), order[len(order)//2].item(), order[-1].item()]
    labels = ['Best', 'Median', 'Worst']

    h = min(n_cols, planner.horizon)
    fig, axes = plt.subplots(3, h, figsize=(h * 2, 6))
    for row, (idx, label) in enumerate(zip(idxs, labels)):
        states = trajectories[idx, :h]
        decoded = symexp(rssm.decoder(states)).clamp(0, 1).cpu().numpy()
        for t in range(h):
            axes[row, t].imshow(np.transpose(decoded[t], (1, 2, 0)))
            axes[row, t].axis('off')
            if t == 0:
                axes[row, t].set_title(f'{label}\nscore={scores[idx]:.2f}', fontsize=9)

    plt.suptitle(f'Decoded prior rollouts (64 candidates, heuristic 80/10/10)', fontsize=11)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight', dpi=120)
    plt.show()


os.makedirs('plots', exist_ok=True)
visualize_candidates(rssm, planner, env, action_size, n_cols=15, save_path='plots/plan_candidates.png')

## 5. Planner in action

Run a full episode with the planner selecting actions at every step.
Show real observations, chosen action, and cumulative reward.

In [ ]:
def planner_policy(full_state, obs):
    rec = full_state[:, :rssm.recurrent_size]
    lat = full_state[:, rssm.recurrent_size:]
    return planner.plan_action(rec, lat)


score, steps = collect_episode(env, rssm, rssm.buffer, action_fn=planner_policy)
print(f'Planner episode: reward={score:.2f}, steps={steps}')

In [ ]:
# Visualise the episode observations
data = rssm.buffer.sample(1, min(steps + 1, rssm.buffer.capacity))
# Find the last episode: scan backwards from the end for a done flag
dones = data['dones'][0].cpu().numpy()
if dones.any():
    end = np.where(dones)[0][-1] + 1
else:
    end = len(dones)
obs_seq = data['observations'][0, :end].cpu().numpy()
rewards_seq = data['rewards'][0, :end].cpu().numpy()

fig, axes = plt.subplots(2, len(obs_seq), figsize=(len(obs_seq) * 1.5, 3))
if len(obs_seq) == 1:
    axes = axes.reshape(2, 1)
for t in range(len(obs_seq)):
    axes[0, t].imshow(np.transpose(obs_seq[t], (1, 2, 0)))
    axes[0, t].axis('off')
    if t == 0:
        axes[0, t].set_title('Observations', fontsize=8)
    axes[1, t].text(0.5, 0.5, f'{rewards_seq[t]:.1f}', ha='center', va='center', fontsize=8)
    axes[1, t].axis('off')
    if t == 0:
        axes[1, t].set_title('Reward', fontsize=8)
plt.suptitle(f'Planner episode: total reward = {score:.2f}, steps = {steps}')
plt.tight_layout()
plt.show()

## 6. Evaluation: Planner vs Random

Compare episode rewards between the CLIP-ViT planner and a random policy.

In [ ]:
def random_policy(state, obs=None):
    valid = getattr(env_eval, 'valid_actions', lambda: list(range(action_size)))()
    idx = np.random.choice(valid)
    return torch.nn.functional.one_hot(
        torch.tensor(idx, device=device).unsqueeze(0), action_size
    ).float()

num_ep = 10

print('Evaluating planner...')
planner_avg, planner_std = evaluate(env_eval, rssm, planner_policy, num_episodes=num_ep)
print(f'Planner: {planner_avg:.2f} +/- {planner_std:.2f}')

print('Evaluating random...')
rand_avg, rand_std = evaluate(env_eval, rssm, random_policy, num_episodes=num_ep)
print(f'Random: {rand_avg:.2f} +/- {rand_std:.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Planner (CLIP-ViT)', 'Random'], [planner_avg, rand_avg],
       yerr=[planner_std, rand_std], capsize=6, color=['#2ecc71', '#95a5a6'])
ax.set_ylabel('Episode reward')
ax.set_title(f'Evaluation over {num_ep} episodes')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('plots/plan_vs_random.png', bbox_inches='tight', dpi=120)
plt.show()

## 7. Distribution of CLIP scores across candidates

For several environment steps, show the histogram of CLIP cosine-similarity scores over all 64 heuristic candidates.

In [ ]:
@torch.no_grad()
def collect_score_distribution(rssm, planner, observation, action, n_steps=5):
    rec, lat = rssm.reset_state()
    enc = rssm.encoder(torch.from_numpy(observation).float().unsqueeze(0).to(rssm.device))
    rec = rssm.recurrent_model(rec, lat, action)
    lat, _ = rssm.posterior_net(torch.cat((rec, enc.view(1, -1)), -1))
    candidates = planner._candidate_sampler.sample()
    trajectories = rssm.imagine_rollouts(rec, lat, candidates)
    scores = clip_scorer.score_rollouts(trajectories, planner.gamma)
    return scores.cpu().numpy()


obs = env_eval.reset()
all_scores = []
act = torch.zeros(1, action_size, device=device)
for step in range(8):
    scores = collect_score_distribution(rssm, planner, obs, act)
    all_scores.append(scores)
    act = planner_policy(torch.zeros(1, rssm.recurrent_size + rssm.latent_size, device=device), obs)
    obs, reward, done, _ = env_eval.step(
        int(torch.argmax(act.cpu().view(-1)).item())
    )  # simplified step for visualisation only

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    if i < len(all_scores):
        ax.hist(all_scores[i], bins=20, color='purple', alpha=0.7)
        ax.set_title(f'Step {i}')
        ax.set_xlabel('CLIP score')
        ax.axvline(all_scores[i].max(), color='red', ls='--', label=f'best={all_scores[i].max():.2f}')
        ax.legend(fontsize=7)
plt.suptitle('Distribution of CLIP scores across 64 heuristic candidates per step')
plt.tight_layout()
plt.savefig('plots/plan_score_distribution.png', bbox_inches='tight', dpi=120)
plt.show()

## 8. Cleanup

In [ ]:
env.close()
env_eval.close()
print('Done.')